In [ ]:
from pathlib import Path

from data_loading import sync_open_data_sessions

In [ ]:
SUBJECT_IDS = ["841312", "841299", "866063", "864846", "864845"]
START_DATE = "2026-06-01"
OUTPUT_ROOT = Path("./data")
if False:
    sync_open_data_sessions(
        subject_ids=SUBJECT_IDS,
        start_date=START_DATE,
        output_root=OUTPUT_ROOT,
        confirm=False,
    )
    !uv run python process_sessions.py

In [ ]:
import pandas as pd
import numpy as np
from helpers import assign_blocks, trim_sessions

trials = pd.read_parquet("data/processed/trials.parquet")
trials = assign_blocks(trials)

# Drop sessions shorter than 15 minutes (first to last site timestamp)
session_start = trials.groupby("session_id")["start_time"].min()
session_end = trials.groupby("session_id")["start_time"].max()
session_duration = session_end - session_start
# Handle both timedelta and numeric (seconds) dtypes
threshold = pd.Timedelta(minutes=15) if pd.api.types.is_timedelta64_dtype(session_duration) else 15 * 60
long_sessions = session_duration[session_duration >= threshold].index
trials = trials[trials["session_id"].isin(long_sessions)]

trials = trim_sessions(trials, start_frac=0.0, end_frac=0.7)


def is_rewarded(patch_label: str) -> bool:
    return "NonRewarded" not in patch_label


def odor_index(odor_concentration: list[float] | np.ndarray) -> int:
    return np.argmax(np.array(odor_concentration))


trials["is_rewarded_odor"] = trials["patch_label"].apply(is_rewarded)
trials["odor_index"] = trials["odor_concentration"].apply(odor_index)


In [ ]:
from helpers import plot_choice_by_block_position
from viz_helpers import a_lot_of_style
from matplotlib import pyplot as plt

for animal in SUBJECT_IDS:
    print(f"Animal {animal}")
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    with a_lot_of_style():
        plot_choice_by_block_position(
            trials[trials["session_id"].str.startswith(animal)], ax=ax
        )

plt.show()

In [ ]:
from helpers import plot_choice_by_block_position_per_session

with a_lot_of_style():
    plot_choice_by_block_position_per_session(trials)

plt.show()

In [ ]:
from helpers import plot_choice_by_block_position_by_first_stop

with a_lot_of_style():
    plot_choice_by_block_position_by_first_stop(trials, single_axis=True)

plt.show()


In [ ]:
from helpers import plot_choice_by_block_position_by_first_stop_overlay

with a_lot_of_style():
    plot_choice_by_block_position_by_first_stop_overlay(trials)

plt.show()

In [ ]:
from helpers import plot_choice_difference_by_block_position_overlay

with a_lot_of_style():
    plot_choice_difference_by_block_position_overlay(trials)

plt.show()

In [ ]:
from helpers import plot_patch_type_delta_heatmap

with a_lot_of_style():
    plot_patch_type_delta_heatmap(trials)

plt.show()

In [ ]:
from helpers import plot_odor_preference_ranking

with a_lot_of_style():
    plot_odor_preference_ranking(trials)

plt.show()

In [ ]:

# P(choice) at the very first trial of every block, averaged within session, plotted across sessions

rs = trials[(trials["site_label"] == "RewardSite") & trials["block"].notna()].copy()
rs = rs.sort_values(["session_id", "block", "start_time"])

rs["_block_pos"] = rs.groupby(["session_id", "block"]).cumcount()

# Add subject and session date label for plotting
rs["subject_id"] = rs["session_id"].str.split("_").str[0]
rs["session_date"] = rs["session_id"].str.split("_").str[1]

STOP_STYLES = {
    0: {"label": "1st stop", "color": "tab:blue"},
    1: {"label": "2nd stop", "color": "tab:orange"},
    2: {"label": "3rd stop", "color": "tab:green"},
}

with a_lot_of_style():
    fig, axes = plt.subplots(1, len(SUBJECT_IDS), figsize=(5 * len(SUBJECT_IDS), 4), sharey=True, squeeze=False)

    for ax, animal in zip(axes[0], SUBJECT_IDS):
        for stop_pos, style in STOP_STYLES.items():
            stop_trials = rs[rs["_block_pos"] == stop_pos].copy()
            session_means = (
                stop_trials.groupby(["subject_id", "session_id", "session_date"])["has_choice"]
                .mean()
                .reset_index()
            )
            sub = session_means[session_means["subject_id"] == animal].sort_values("session_id")
            x = np.arange(len(sub))
            ax.plot(x, sub["has_choice"], marker="o", color=style["color"], label=style["label"])

        # Use session dates from 1st stop for x-axis labels (most sessions should have it)
        first_stop_sub = rs[(rs["_block_pos"] == 0) & (rs["subject_id"] == animal)]
        dates = sorted(first_stop_sub["session_id"].unique())
        date_labels = [s.split("_")[1] for s in dates]
        ax.set_xticks(np.arange(len(dates)))
        ax.set_xticklabels(date_labels, rotation=45, ha="right")
        ax.set_xlabel("Session")
        ax.set_ylabel("P(choice)")
        ax.set_ylim(0, 1.05)
        ax.set_title(f"Subject {animal}")
        ax.legend(frameon=False, fontsize=8)

    fig.suptitle("P(choice) at 1st, 2nd, and 3rd trial of each block (session averages)")
    fig.tight_layout()

plt.show()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from viz_helpers import a_lot_of_style

# ── Build within-block, 1-trial-back feature table ──────────────────────────
rs = trials[(trials["site_label"] == "RewardSite") & trials["block"].notna()].copy()
rs = rs.sort_values(["session_id", "block", "start_time"])

grp = rs.groupby(["session_id", "block"], sort=False)
rs["prev_odor_index"] = grp["odor_index"].shift(1)
rs["prev_has_choice"] = grp["has_choice"].shift(1)
rs["prev_has_reward"]  = grp["has_reward"].shift(1)

rs = rs.dropna(subset=["prev_odor_index", "prev_has_choice", "prev_has_reward"])

# ── Regressors ───────────────────────────────────────────────────────────────
is_same       = (rs["odor_index"] == rs["prev_odor_index"]).to_numpy()
prev_choice   = rs["prev_has_choice"].astype(bool).to_numpy()
prev_rewarded = rs["prev_has_reward"].astype(bool).to_numpy()

rs["IsPrevChoice_SameOdor"]  = np.where(is_same,  np.where(prev_choice, 1.0, -1.0), 0.0)
rs["IsPrevChoice_OtherOdor"] = np.where(~is_same, np.where(prev_choice, 1.0, -1.0), 0.0)

rs["H_Same_Rew"]    = ( is_same &  prev_rewarded).astype(float)
rs["H_Same_NoRew"]  = ( is_same & ~prev_rewarded).astype(float)
rs["H_Other_Rew"]   = (~is_same &  prev_rewarded).astype(float)
rs["H_Other_NoRew"] = (~is_same & ~prev_rewarded).astype(float)

rs["choice"] = rs["has_choice"].astype(int)

PLOT_COEFS = [
    "IsPrevChoice_SameOdor",
    "IsPrevChoice_OtherOdor",
    "H_Same_Rew",
    "H_Same_NoRew",
    "H_Other_Rew",
    "H_Other_NoRew",
]
FEATURE_COLS = PLOT_COEFS

# ── Fit logistic GLM per session ─────────────────────────────────────────────
records = []
for session_id, sdf in rs.groupby("session_id"):
    if len(sdf) < 10 or sdf["choice"].nunique() < 2:
        continue
    X = sdf[FEATURE_COLS].to_numpy(dtype=float)
    y = sdf["choice"].to_numpy(dtype=int)
    try:
        clf = LogisticRegression(C=np.inf, solver="lbfgs", fit_intercept=False, max_iter=500)
        clf.fit(X, y)
        for name, val in zip(PLOT_COEFS, clf.coef_[0]):
            records.append(dict(
                session_id=session_id,
                subject_id=session_id.split("_")[0],
                coef=name,
                value=val,
            ))
    except Exception as e:
        print(f"Session {session_id} failed: {e}")

coefs = pd.DataFrame(records)

# ── Colour scheme ─────────────────────────────────────────────────────────────
SAME_COLOR    = "#e07b39"
OTHER_COLOR   = "#4f8fc0"
NEUTRAL_COLOR = "gray"

def _coef_color(name):
    if name.startswith("H_Same"):  return SAME_COLOR
    if name.startswith("H_Other"): return OTHER_COLOR
    if "Same" in name:             return SAME_COLOR
    if "Other" in name:            return OTHER_COLOR
    return NEUTRAL_COLOR

TICK_LABELS = [
    "IsPrevChoice\n[same]",
    "IsPrevChoice\n[other]",
    "Same × Rew",
    "Same × NoRew",
    "Other × Rew",
    "Other × NoRew",
]

subjects  = sorted(coefs["subject_id"].unique())
x_pos     = np.arange(len(PLOT_COEFS))
rng       = np.random.default_rng(0)

PAIR_GROUPS = [
    (["IsPrevChoice_SameOdor", "IsPrevChoice_OtherOdor"], "#f5f5f5"),
    (["H_Same_Rew", "H_Same_NoRew"],   "#fff3eb"),
    (["H_Other_Rew", "H_Other_NoRew"], "#ebf3ff"),
]

N_BOOTSTRAP = 2000

with a_lot_of_style():
    fig, axes = plt.subplots(
        2, len(subjects),
        figsize=(8 * len(subjects), 10),
        sharey="row",
        squeeze=False,
    )

    for col, subject in enumerate(subjects):
        # ── Row 0: per-session scatter + mean±SEM ────────────────────────────
        ax = axes[0][col]
        for group_members, bg in PAIR_GROUPS:
            idxs = [PLOT_COEFS.index(m) for m in group_members]
            ax.axvspan(min(idxs) - 0.4, max(idxs) + 0.4, color=bg, zorder=0)

        sub      = coefs[coefs["subject_id"] == subject]
        sessions = sorted(sub["session_id"].unique())
        n        = len(sessions)
        cmap     = plt.get_cmap("viridis")
        norm     = Normalize(vmin=0, vmax=max(n - 1, 1))

        for day, session_id in enumerate(sessions):
            sdata = sub[sub["session_id"] == session_id].set_index("coef")["value"]
            vals  = [sdata.get(c, np.nan) for c in PLOT_COEFS]
            jx    = x_pos + rng.uniform(-0.15, 0.15, len(x_pos))
            ax.scatter(jx, vals, color=cmap(norm(day)), s=40, zorder=3, alpha=0.85)

        means = sub.groupby("coef")["value"].mean().reindex(PLOT_COEFS)
        sems  = sub.groupby("coef")["value"].sem().reindex(PLOT_COEFS)
        for xi, coef in enumerate(PLOT_COEFS):
            ax.errorbar(xi, means[coef], yerr=sems[coef],
                        fmt="o", color=_coef_color(coef), ms=8, lw=2.5, capsize=5, zorder=5)

        ax.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.7)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(TICK_LABELS, rotation=30, ha="right", fontsize=9)
        ax.set_title(f"Subject {subject}")
        ax.set_xlabel("Regressor")

        cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax)
        cb.set_label("Session (day)")
        cb.set_ticks([0, max(n - 1, 1)])
        cb.set_ticklabels(["first", "last"])

        # ── Row 1: bootstrapped mean ± 95% CI across sessions ────────────────
        ax2 = axes[1][col]
        for group_members, bg in PAIR_GROUPS:
            idxs = [PLOT_COEFS.index(m) for m in group_members]
            ax2.axvspan(min(idxs) - 0.4, max(idxs) + 0.4, color=bg, zorder=0)

        # Build session-level coefficient matrix: (n_sessions, n_coefs)
        session_list = sorted(sub["session_id"].unique())
        coef_matrix = np.array([
            [sub[sub["session_id"] == sid].set_index("coef")["value"].reindex(PLOT_COEFS).values
             for sid in session_list]
        ]).squeeze(0)  # shape: (n_sessions, n_coefs)

        observed_mean = np.nanmean(coef_matrix, axis=0)

        boot_rng = np.random.default_rng(42)
        boot_means = np.array([
            np.nanmean(coef_matrix[boot_rng.integers(0, len(session_list), size=len(session_list))], axis=0)
            for _ in range(N_BOOTSTRAP)
        ])  # shape: (N_BOOTSTRAP, n_coefs)

        ci_lo = np.nanpercentile(boot_means, 2.5, axis=0)
        ci_hi = np.nanpercentile(boot_means, 97.5, axis=0)

        for xi, coef in enumerate(PLOT_COEFS):
            color = _coef_color(coef)
            ax2.bar(xi, observed_mean[xi], color=color, alpha=0.75, width=0.6, zorder=2)
            ax2.errorbar(
                xi, observed_mean[xi],
                yerr=[[observed_mean[xi] - ci_lo[xi]], [ci_hi[xi] - observed_mean[xi]]],
                fmt="none", color="black", capsize=5, lw=1.5, zorder=3,
            )

        ax2.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.7)
        ax2.set_xticks(x_pos)
        ax2.set_xticklabels(TICK_LABELS, rotation=30, ha="right", fontsize=9)
        ax2.set_title(f"Subject {subject} — bootstrap mean")
        ax2.set_xlabel("Regressor")

    axes[0][0].set_ylabel("GLM coefficient (per session)")
    axes[1][0].set_ylabel("GLM coefficient (bootstrap mean ± 95% CI)")
    fig.suptitle("Logistic GLM: P(choice) — within-block, 1-trial history\n"
                 "Reward encoding: one-hot (is_same × is_prev_rewarded), no intercept")
    fig.tight_layout()

plt.show()


In [ ]:

# ── One-hot reward cells timecourse across sessions ───────────────────────────
# 4 distinct colors so solid/dashed ambiguity is avoided entirely
REWARD_CELLS = {
    "H_Same_Rew":    {"label": "Same × Rew",    "color": "#e07b39", "marker": "o"},
    "H_Same_NoRew":  {"label": "Same × NoRew",  "color": "#f5c18a", "marker": "o"},
    "H_Other_Rew":   {"label": "Other × Rew",   "color": "#2a6496", "marker": "s"},
    "H_Other_NoRew": {"label": "Other × NoRew", "color": "#9ecae1", "marker": "s"},
}

with a_lot_of_style():
    fig, axes = plt.subplots(
        1, len(subjects),
        figsize=(6 * len(subjects), 4),
        sharey=True,
        squeeze=False,
    )

    for ax, subject in zip(axes[0], subjects):
        sub      = coefs[coefs["subject_id"] == subject]
        sessions = sorted(sub["session_id"].unique())
        x        = np.arange(len(sessions))
        dates    = [s.split("_")[1] for s in sessions]

        for term, style in REWARD_CELLS.items():
            rows = sub[sub["coef"] == term].set_index("session_id")
            vals = [rows.loc[sid, "value"] if sid in rows.index else np.nan for sid in sessions]
            ax.plot(x, vals,
                    marker=style["marker"], color=style["color"],
                    linewidth=2, markersize=7, label=style["label"])

        ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(dates, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Session")
        ax.set_title(f"Subject {subject}")
        ax.legend(frameon=False, fontsize=8)

    axes[0][0].set_ylabel("GLM coefficient")
    fig.suptitle("One-hot reward cells timecourse")
    fig.tight_layout()

plt.show()


In [ ]:

# ── Cross-block odor bias: 4-condition analysis ───────────────────────────────
#
# For each odor, track its consecutive block appearances chronologically.
# Each consecutive pair (block N -> next block where odor appears) gives:
#   prev_rewarded = was the odor the rewarded one in block N?
#   curr_rewarded = is  the odor the rewarded one in the next block?
#   p_stop_curr   = mean P(stop) for that odor in the next block
#
# 4 conditions: (prev rewarded / not) x (curr rewarded / not) -> P(stop) +- SEM
# One figure per animal.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from viz_helpers import a_lot_of_style

rs = trials[(trials["site_label"] == "RewardSite") & trials["block"].notna()].copy()
rs["subject_id"] = rs["session_id"].str.split("_").str[0]

# Per (subject, session, block, odor) summary
odor_block = (
    rs.groupby(["subject_id", "session_id", "block", "odor_index"])
    .agg(
        p_stop=("has_choice", "mean"),
        n_trials=("has_choice", "count"),
        is_rewarded_odor=("is_rewarded_odor", "first"),
    )
    .reset_index()
)
# session_id encodes datetime -> lexicographic sort is chronological
odor_block = odor_block.sort_values(
    ["subject_id", "odor_index", "session_id", "block"]
)

# Build consecutive block-appearance pairs per (subject, odor)
pair_records = []
for (subject, odor), grp in odor_block.groupby(["subject_id", "odor_index"]):
    grp = grp.reset_index(drop=True)
    for i in range(len(grp) - 1):
        prev = grp.iloc[i]
        curr = grp.iloc[i + 1]
        pair_records.append({
            "subject_id": subject,
            "odor_index": int(odor),
            "prev_rewarded": bool(prev["is_rewarded_odor"]),
            "curr_rewarded": bool(curr["is_rewarded_odor"]),
            "p_stop_curr": curr["p_stop"],
            "n_trials_curr": int(curr["n_trials"]),
        })

pairs_df = pd.DataFrame(pair_records)

# ── Plot ──────────────────────────────────────────────────────────────────────
CONDITIONS = [
    (True,  True,  "Prev Rew\n-> Curr Rew",    "#c0392b"),
    (True,  False, "Prev Rew\n-> Curr NoRew",  "#e07b39"),
    (False, True,  "Prev NoRew\n-> Curr Rew",  "#1a5276"),
    (False, False, "Prev NoRew\n-> Curr NoRew","#4f8fc0"),
]

subjects = sorted(pairs_df["subject_id"].unique())

with a_lot_of_style():
    fig, axes = plt.subplots(
        1, len(subjects),
        figsize=(5 * len(subjects), 5),
        sharey=True,
        squeeze=False,
    )
    fig.suptitle(
        "P(stop) at next odor encounter — 4 conditions\n"
        "(prev block rewarded / not) x (curr block rewarded / not)",
        fontsize=10,
    )

    for ai, subject in enumerate(subjects):
        ax = axes[0][ai]
        sub = pairs_df[pairs_df["subject_id"] == subject]

        ax.axvspan(-0.5, 1.5, color="#fff0eb", zorder=0)
        ax.axvspan(1.5, 3.5, color="#eaf3fb", zorder=0)

        for xi, (prev_rew, curr_rew, label, color) in enumerate(CONDITIONS):
            grp = sub[
                (sub["prev_rewarded"] == prev_rew) &
                (sub["curr_rewarded"] == curr_rew)
            ]["p_stop_curr"]
            n = len(grp)
            m = grp.mean() if n > 0 else float("nan")
            se = grp.sem() if n > 1 else 0.0

            ax.bar(xi, m, color=color, alpha=0.85, width=0.65, zorder=2)
            if m == m:  # not nan
                ax.errorbar(xi, m, yerr=se, fmt="none", color="black",
                            capsize=5, lw=1.5, zorder=3)
                ax.text(xi, min(m + se + 0.04, 1.18), f"n={n}",
                        ha="center", va="bottom", fontsize=7, zorder=4)

        ax.axvline(1.5, color="black", lw=1.0, alpha=0.3, zorder=1)
        ax.axhline(0.5, color="gray", linestyle="--", lw=0.8, alpha=0.5)

        ax.set_xticks(range(len(CONDITIONS)))
        ax.set_xticklabels([c[2] for c in CONDITIONS], fontsize=8)
        ax.set_ylim(0, 1.35)
        ax.set_title(f"Subject {subject}", fontsize=10)

        if ai == 0:
            ax.set_ylabel("P(stop) in next block encounter")

        ax.text(0.5, 1.28, "Prev: Rewarded", ha="center", va="top",
                fontsize=7.5, color="#8b0000",
                transform=ax.get_xaxis_transform())
        ax.text(2.5, 1.28, "Prev: Not Rewarded", ha="center", va="top",
                fontsize=7.5, color="#1a5276",
                transform=ax.get_xaxis_transform())

    fig.tight_layout()
    plt.show()
